# 🧬 Synthetic Medical Data Generation
This notebook demonstrates how to use open-source LLMs to generate realistic synthetic medical datasets for privacy-preserving research, modeling, or training.

In [ ]:

!pip install pandas faker transformers datasets


In [ ]:

import pandas as pd
from faker import Faker
import random

from transformers import pipeline

# Set up faker and seed
fake = Faker()
random.seed(42)
Faker.seed(42)


In [ ]:

def generate_fake_patient_record():
    return {
        "PatientID": fake.uuid4(),
        "Name": fake.name(),
        "Age": random.randint(0, 100),
        "Sex": random.choice(["M", "F"]),
        "DateOfVisit": fake.date_between(start_date='-2y', end_date='today'),
        "Symptoms": fake.sentence(nb_words=6),
        "Diagnosis": random.choice(["Hypertension", "Diabetes", "Asthma", "COVID-19", "Heart Failure", "Healthy"]),
        "Medications": fake.words(nb=2),
        "DoctorNotes": fake.text(max_nb_chars=150)
    }

# Generate 100 rows
data = [generate_fake_patient_record() for _ in range(100)]
df = pd.DataFrame(data)
df.head()


In [ ]:

df.to_csv("synthetic_patient_data.csv", index=False)
print("✅ Saved as synthetic_patient_data.csv")


In [ ]:

from transformers import pipeline

# Using a general model to enhance text, can be replaced with medical-specific models like BioGPT or Meditron
enhancer = pipeline("text2text-generation", model="google/flan-t5-base")

example = df.iloc[0]["DoctorNotes"]
enhancer(f"Improve clinical coherence of: {example}", max_length=100)[0]["generated_text"]



---
✅ **Generated:** 2025-05-30 11:32:37  
- 100 synthetic patient records with demographics, diagnosis, symptoms, and notes  
- Notes optionally enhanced with a text-to-text model for realism  
- Saved as CSV for modeling or training


## Gradio UI for Synthetic Data Generation

In [ ]:

import gradio as gr

def generate_data_ui(count):
    data = generate_synthetic_data(count)
    return data.to_csv(index=False)

gr.Interface(
    fn=generate_data_ui,
    inputs=gr.Number(label="Number of Patients", value=5),
    outputs=gr.File(label="Generated Data CSV"),
    title="Synthetic Medical Data Generator",
    description="Generate synthetic patient records for ML model training or testing."
).launch()


## Export to FHIR/HL7 or RDF Format

In [ ]:

from rdflib import Graph, Literal, RDF, URIRef, Namespace

def export_to_rdf(df):
    ns = Namespace("http://example.org/health#")
    g = Graph()
    for _, row in df.iterrows():
        patient_uri = URIRef(f"http://example.org/patient/{row['patient_id']}")
        g.add((patient_uri, RDF.type, ns.Patient))
        g.add((patient_uri, ns.age, Literal(row['age'])))
        g.add((patient_uri, ns.gender, Literal(row['gender'])))
        g.add((patient_uri, ns.diagnosis, Literal(row['diagnosis'])))
        g.add((patient_uri, ns.lab_result, Literal(row['lab_result'])))
    g.serialize("synthetic_patients.ttl", format="turtle")
    return "synthetic_patients.ttl"

# Export example
export_to_rdf(generate_synthetic_data(10))


## Chain with Downstream ML Models

In [ ]:

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

df = generate_synthetic_data(200)
X = df[['age', 'lab_result']]
y = df['diagnosis'].map(lambda x: 1 if x == 'diabetes' else 0)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3)

model = LogisticRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print(classification_report(y_test, y_pred))
